# The same rules, written as functions

Two functions are enough to check the hand work. `gini` is the disagreement score. `best_split` tries every midpoint, then applies the published tie-break: larger gain, then the earlier feature, then the smaller threshold. `grow` stops when the leaf is pure or the best gain is $0$. `predict` walks whatever tree `grow` returned.

On the eight parts this must reproduce length $\le 7/2$, then weight $\le 3$, then a fail leaf. It must also refuse the first XOR cut, because that cut's gain is $0$.


In [1]:
# A readable grower. It stops when the best gain is not positive.
from fractions import Fraction

def gini(labels):
    n = len(labels)
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return 1 - sum(Fraction(count, n) ** 2 for count in counts.values())

def best_split(rows):
    parent = gini([row[-1] for row in rows])
    winner = None
    n_features = len(rows[0]) - 1
    for feature in range(n_features):
        values = sorted({row[feature] for row in rows})
        for left_value, right_value in zip(values, values[1:]):
            threshold = Fraction(left_value + right_value, 2)
            left = [row for row in rows if row[feature] <= threshold]
            right = [row for row in rows if row[feature] > threshold]
            weighted = (
                Fraction(len(left), len(rows)) * gini([row[-1] for row in left])
                + Fraction(len(right), len(rows)) * gini([row[-1] for row in right])
            )
            gain = parent - weighted
            candidate = (gain, -feature, -threshold, feature, threshold)
            if winner is None or candidate[:3] > winner[:3]:
                winner = candidate
    return winner

def grow(rows):
    labels = [row[-1] for row in rows]
    if gini(labels) == 0:
        return {"leaf": labels[0]}
    winner = best_split(rows)
    if winner is None or winner[0] <= 0:
        # Tie-break for a mixed leaf that will not be split: prefer pass.
        chosen = "pass" if labels.count("pass") >= labels.count("fail") else "fail"
        return {"leaf": chosen}
    gain, _, _, feature, threshold = winner
    left = [row for row in rows if row[feature] <= threshold]
    right = [row for row in rows if row[feature] > threshold]
    return {
        "feature": feature,
        "threshold": threshold,
        "gain": gain,
        "left": grow(left),
        "right": grow(right),
    }

def predict(tree, measurements):
    if "leaf" in tree:
        return tree["leaf"]
    if measurements[tree["feature"]] <= tree["threshold"]:
        return predict(tree["left"], measurements)
    return predict(tree["right"], measurements)

parts = [
    (1, 1, "pass"), (1, 2, "pass"), (2, 1, "pass"), (2, 2, "pass"),
    (1, 4, "fail"), (2, 4, "fail"), (5, 1, "fail"), (5, 2, "fail"),
]
tree = grow(parts)
assert tree["feature"] == 0
assert tree["threshold"] == Fraction(7, 2)
assert tree["gain"] == Fraction(1, 6)
assert tree["left"]["feature"] == 1
assert tree["left"]["threshold"] == 3
assert tree["left"]["gain"] == Fraction(4, 9)
assert tree["right"] == {"leaf": "fail"}
for length, weight, label in parts:
    assert predict(tree, (length, weight)) == label
xor = [(0, 0, "fail"), (0, 1, "pass"), (1, 0, "pass"), (1, 1, "fail")]
xor_tree = grow(xor)
assert "leaf" in xor_tree
print("root", tree["feature"], tree["threshold"], tree["gain"])
print("XOR stopped at", xor_tree)


root 0 7/2 1/6
XOR stopped at {'leaf': 'pass'}


## Force the zero-gain cut, and XOR separates

The function above obeyed the stop rule. A second function can keep a zero-gain cut when a depth budget remains. Depth $2$ is enough for these four points. The root feature is $0$ because of the tie-break, the threshold is $1/2$, and both children become pure after one question on feature $1$.


In [2]:
# Same splitter, but a zero gain is allowed until the depth budget ends.
from fractions import Fraction

def gini(labels):
    n = len(labels)
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return 1 - sum(Fraction(count, n) ** 2 for count in counts.values())

def best_split(rows):
    parent = gini([row[-1] for row in rows])
    winner = None
    for feature in range(len(rows[0]) - 1):
        values = sorted({row[feature] for row in rows})
        for left_value, right_value in zip(values, values[1:]):
            threshold = Fraction(left_value + right_value, 2)
            left = [row for row in rows if row[feature] <= threshold]
            right = [row for row in rows if row[feature] > threshold]
            weighted = (
                Fraction(len(left), len(rows)) * gini([row[-1] for row in left])
                + Fraction(len(right), len(rows)) * gini([row[-1] for row in right])
            )
            candidate = (parent - weighted, -feature, -threshold, feature, threshold)
            if winner is None or candidate[:3] > winner[:3]:
                winner = candidate
    return winner

def grow(rows, depth):
    labels = [row[-1] for row in rows]
    if gini(labels) == 0 or depth == 0:
        chosen = "pass" if labels.count("pass") >= labels.count("fail") else "fail"
        return {"leaf": labels[0] if gini(labels) == 0 else chosen}
    _, _, _, feature, threshold = best_split(rows)
    left = [row for row in rows if row[feature] <= threshold]
    right = [row for row in rows if row[feature] > threshold]
    return {
        "feature": feature,
        "threshold": threshold,
        "left": grow(left, depth - 1),
        "right": grow(right, depth - 1),
    }

def predict(tree, measurements):
    if "leaf" in tree:
        return tree["leaf"]
    branch = "left" if measurements[tree["feature"]] <= tree["threshold"] else "right"
    return predict(tree[branch], measurements)

xor = [(0, 0, "fail"), (0, 1, "pass"), (1, 0, "pass"), (1, 1, "fail")]
tree = grow(xor, depth=2)
assert tree["feature"] == 0 and tree["threshold"] == Fraction(1, 2)
for x, y, label in xor:
    assert predict(tree, (x, y)) == label
print("XOR root", tree["feature"], tree["threshold"])


XOR root 0 1/2


## A pitfall

The stop rule is part of the model. Two functions with the same impurity can return different trees on XOR: a single leaf, or four pure leaves. When you report a tree, report whether a gain of $0$ was allowed to split.

## What to carry forward

The grower recovers the hand tree on the eight parts: length $\le 7/2$ with gain $1/6$, then weight $\le 3$ with gain $4/9$. On XOR it stops at the root unless the depth budget forces the zero-gain cut.
